# 7.2 — Q-Learning: تحلیل کد اشتباه و کد صحیح

## Grid-World

- حالت‌ها: A, B, C, D, E, F
- حالت جاذب (Absorbing State): **F**
- پاداش‌های فوری: انتقال در حالت‌های B، D و F
- ضریب تخفیف: **gamma = 0.9**

```
+----------+----------+----------+
|    A  ---5-->  D  -->   E      |
|    |          ^         |      |
|   10          5        20      |
|    v          |         v      |
|    B  -->    C  --20-->  F <---+
+----------+----------+----------+
```

## فرمول Q-Learning

Q-Learning یک الگوریتم **off-policy** یادگیری تقویتی است.

فرمول آپدیت:

$$Q(s, a) \leftarrow r(s, a) + \gamma \cdot \max_{a'} Q(s', a')$$

| نماد | معنی |
|------|------|
| $s$ | حالت فعلی |
| $a$ | اکشن انجام‌شده |
| $s'$ | حالت بعدی |
| $r(s,a)$ | پاداش فوری برای انتقال از $s$ با اکشن $a$ |
| $\gamma$ | ضریب تخفیف |
| $\max_{a'} Q(s', a')$ | بیشترین Q-value در حالت بعدی |

### توابع کمکی

- `apply_random_action(state)` — یک اکشن تصادفی اجرا می‌کند و `(action, new_state)` برمی‌گرداند
- `get_reward(state)` — پاداش فوری انتقال **از** `state` را برمی‌گرداند
- `is_goal_state(state)` — اگر `state` حالت جاذب باشد `True` برمی‌گرداند
- `update(action, action_reward)` — Q-value اکشن را در جدول آپدیت می‌کند
- `get_max_q(state)` — ماکزیمم Q-value برای همه اکشن‌های ممکن از `state` برمی‌گرداند

---
## کد اشتباه (Incorrect)

کد زیر مستقیماً از صورت تمرین گرفته شده است و دارای **دو خطا** است.

In [ ]:
# ============================================================
# کد اشتباه — دارای دو خطای اساسی
# ============================================================

def q_learn_WRONG(state, gamma):
    # یک اکشن تصادفی انجام می‌دهیم
    a, new_state = apply_random_action(state)

    # WRONG line 3: get_reward با new_state صدا زده می‌شود
    # پاداش فوری مربوط به انتقال از state فعلی است نه new_state
    imm_reward = get_reward(new_state)    # <--- WRONG

    if is_goal_state(new_state):
        a_reward = imm_reward
    else:
        # WRONG line 8: فراخوانی بازگشتی تصادفی به جای max Q-value
        # q_learn_WRONG یک اکشن تصادفی جدید در new_state اجرا می‌کند
        # Q-Learning باید از max Q(s', a') استفاده کند نه یک مسیر تصادفی
        a_reward = imm_reward + gamma * q_learn_WRONG(new_state, gamma)    # <--- WRONG

    update(a, a_reward)
    return a_reward

---
## توضیح خطاها

### خطا ۱ — خط ۳

```python
imm_reward = get_reward(new_state)   # اشتباه
imm_reward = get_reward(state)       # صحیح
```

**دلیل:** پاداش فوری $r(s, a)$ به **transition از state فعلی** تعلق دارد.  
مثال: وقتی از `A` با اکشن راست به `D` می‌روی، پاداش `5` برای **ترک کردن `A`** است.  
پس `get_reward` باید با `state` صدا زده شود، نه `new_state`.

---

### خطا ۲ — خط ۸

```python
a_reward = imm_reward + gamma * q_learn(new_state, gamma)   # اشتباه
a_reward = imm_reward + gamma * get_max_q(new_state)        # صحیح
```

**دلیل:** Q-Learning یک الگوریتم **off-policy** است و از ماکزیمم Q-value استفاده می‌کند:

$$\max_{a'} Q(s', a')$$

اما `q_learn(new_state, gamma)` با `apply_random_action` یک اکشن **تصادفی** جدید اجرا می‌کند:

- اکتشاف غیرضروری انجام می‌دهد (به جای lookup از جدول Q)
- off-policy نیست — از بهترین Q-value استفاده نمی‌کند
- بیشتر شبیه **SARSA** (on-policy) با policy تصادفی است

راه‌حل: استفاده از `get_max_q(new_state)` که مستقیماً $\max_{a'} Q(s', a')$ را از جدول Q برمی‌گرداند.

---
## کد صحیح (Corrected)

دو خطا با توضیح در کد زیر اصلاح شده‌اند.

In [ ]:
# ============================================================
# کد صحیح — Q-Learning (off-policy)
# Q(s, a) = r(s, a) + gamma * max_a Q(s', a')
# ============================================================

def q_learn(state, gamma):
    # یک اکشن تصادفی انجام می‌دهیم و حالت بعدی را دریافت می‌کنیم
    a, new_state = apply_random_action(state)

    # FIX 1: پاداش فوری مربوط به state فعلی است، نه new_state
    # r(s, a): پاداش انتقال از state با اکشن a
    imm_reward = get_reward(state)    # <--- FIXED

    if is_goal_state(new_state):
        # حالت جاذب: Q-value فقط همان پاداش فوری است
        # چون V(F) = 0 و دیگر حالتی بعد از آن وجود ندارد
        # Q(s, a) = r(s, a)
        a_reward = imm_reward
    else:
        # FIX 2: از ماکزیمم Q-value حالت بعدی استفاده می‌کنیم
        # Q(s, a) = r(s, a) + gamma * max_a' Q(s', a')
        # get_max_q مستقیماً از جدول Q می‌خواند — اکشن جدید اجرا نمی‌کند
        a_reward = imm_reward + gamma * get_max_q(new_state)    # <--- FIXED

    # Q-value اکشن a را در جدول Q آپدیت می‌کنیم
    update(a, a_reward)

    return a_reward

---
## خلاصه تفاوت‌ها

| خط | کد اشتباه | کد صحیح | علت |
|----|-----------|---------|-----|
| 3 | `get_reward(new_state)` | `get_reward(state)` | پاداش متعلق به state فعلی است |
| 8 | `q_learn(new_state, gamma)` | `get_max_q(new_state)` | Q-Learning از max Q-value استفاده می‌کند |

---

## تفاوت Q-Learning با SARSA

| ویژگی | Q-Learning (صحیح) | SARSA |
|--------|------------------|-------|
| نوع | Off-policy | On-policy |
| فرمول | $r + \gamma \max_{a'} Q(s', a')$ | $r + \gamma Q(s', a')$ |
| اکشن بعدی | ماکزیمم Q-value از جدول | اکشن واقعی بعدی |
| کد اشتباه | شبیه SARSA با اکشن تصادفی | — |